<a href="https://colab.research.google.com/github/Abrar-404/AI-ML_Practices_and_Assignments/blob/main/Own_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -------------------------------------------------
#  Module 11
# -------------------------------------------------

# 1. Manual Logistic Regression from Scratch

## Dataset: Breast Cancer Wisconsin Dataset

Download:
•	https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data

Tasks
- 1.	Load the dataset and drop any ID / unnamed columns.
- 2.	Split it into 80% training and 20% testing data.
- 3.	Scale the features with StandardScaler and encode the target with LabelEncoder.
- 4.	Convert the data to PyTorch tensors.
- 5.	Implement a single-neuron model by hand: random weights and zero bias (both with requires_grad=True), a sigmoid forward pass, and a binary cross-entropy loss function (no torch.nn, no torch.optim).
- 6.	Train for 25 epochs with a manual gradient-descent update loop, printing the loss every epoch.
- 7.	Evaluate accuracy on the test set.
Does the loss decrease smoothly across all 25 epochs? If not, what might be causing the jumps?


In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [27]:
# Load the dataset and drop any ID / unnamed columns.
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

X = df.drop(['id', 'diagnosis', 'Unnamed: 32'], axis = 1)
y = df['diagnosis']

# Split it into 80% training and 20% testing data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Scale the features with StandardScaler and encode the target with LabelEncoder.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# Convert the data to PyTorch tensors.
X_train_tensor = torch.from_numpy(X_train)
y_train_tensor = torch.from_numpy(y_train).reshape(-1, 1)
X_test_tensor = torch.from_numpy(X_test)
y_test_tensor = torch.from_numpy(y_test).reshape(-1, 1)

# Implement a single-neuron model by hand: random weights and zero bias (both with requires_grad=True), a sigmoid forward pass, and a binary cross-entropy loss function (no torch.nn, no torch.optim).
class SimpleModel():
  def __init__(self, x):
    self.weights = torch.rand(x.shape[1], 1, dtype = torch.float64, requires_grad = True)
    self.bias = torch.zeros(1, dtype = torch.float64, requires_grad = True)

  def forward(self, x):
    z = torch.matmul(x, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_func(self, y_pred, y):
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    loss = -torch.mean(y * torch.log(y_pred) + (1 - y) * (torch.log(1 - y_pred)))
    return loss


# Train for 25 epochs with a manual gradient-descent update loop, printing the loss every epoch.
learning_rate = 0.1
epochs = 25

model = SimpleModel(X_train_tensor)

for epoch in range(epochs):
  # forward pass
  y_pred = model.forward(X_train_tensor)

  # loss calculation
  loss = model.loss_func(y_pred, y_train_tensor)

  # backpropagation
  loss.backward()

  # update weights and bias
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad

  # zero gradient
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  # print loss in epochs
  print(f'epochs: {epoch + 1}, loss: {loss.item()}')


print()
print()
print()

# Evaluate accuracy on the test set. Does the loss decrease smoothly across all 25 epochs? If not, what might be causing the jumps?
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'accuracy: {accuracy.item()}')


# In plain terms:

# The loss should go down smoothly, step by step, without any weird jumps back up — and it did.
# That's because logistic regression's loss has a single bowl-shaped curve (convex) — there's only one minimum, so gradient descent just walks straight downhill toward it, no zigzagging between different "valleys."
# You're also using all the training data at once each step (not small random batches), so there's no randomness making the loss bounce around epoch to epoch.

# If you ever did see the loss jump around instead of decreasing steadily, it would usually mean one of these:

# Learning rate too high — you're taking steps so big you overshoot past the minimum and bounce around it instead of settling in.
# Features not scaled properly — if one feature has huge values compared to others, it dominates the gradient and throws off the updates.
# A code bug, like the shape mismatch earlier — feeding the model bad-shaped data corrupts the gradient calculation and gives you garbage updates.

epochs: 1, loss: 0.5226084517090123
epochs: 2, loss: 0.5075175172042845
epochs: 3, loss: 0.492823537472651
epochs: 4, loss: 0.4785245835193676
epochs: 5, loss: 0.4646204962914867
epochs: 6, loss: 0.4511125004054087
epochs: 7, loss: 0.4380025250284362
epochs: 8, loss: 0.4252922422081844
epochs: 9, loss: 0.41298195226126017
epochs: 10, loss: 0.4010695649403179
epochs: 11, loss: 0.38954997063911667
epochs: 12, loss: 0.37841501250164233
epochs: 13, loss: 0.36765407769923875
epochs: 14, loss: 0.357255123492924
epochs: 15, loss: 0.34666324096732914
epochs: 16, loss: 0.3346876679567407
epochs: 17, loss: 0.3231089950286521
epochs: 18, loss: 0.31191886647698097
epochs: 19, loss: 0.3011116598402604
epochs: 20, loss: 0.290684060035178
epochs: 21, loss: 0.28063433516050407
epochs: 22, loss: 0.27096148447931473
epochs: 23, loss: 0.26166441746780345
epochs: 24, loss: 0.252741294167334
epochs: 25, loss: 0.24418910822308337



accuracy: 0.9649122953414917
